# Grafting 2D tSNE points onto **RELAXED SCAFFOLD** sphere

We have a scaffold projected as a sphere containing order points (130). Now, for each order, we need to take the 2D points (14x) and graft them onto the surface of the scaffold sphere, centered around each order point.

This is identical to the previous notebook, 151, **EXCEPT** we are using the pre-relaxed scaffold sphere.

Basic method:
1. do 2D tSNE plots for all orders
2. Graft these all onto the scaffold sphere
3. Place ones with too few points on at the same location as the order-level point (these have between 1 and 20 points)
4. Run spherical scatterplot relaxation
5. Re-compute order-level point based on centroid of relaxed points
6. re-run integrate_tree_to_xyz based on re-computed order-level points
7. Display everything

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

## The RELAXED Scaffold

This is the initial positioning of the order points as determined from the order-level tree (130), but RELAXED. 

In [ ]:
# Load in the scaffold sphere points.

scaffold_filename = "output/scaffold/order-scaffold-relaxation/order-scaffold-relaxed100.csv"

relaxed_scaffold_df = pd.read_csv(scaffold_filename, index_col=0)

# These are normalized x,y, and z coordinates. Convert them to lat/lon.
relaxed_scaffold_df["lat"] = np.degrees(np.arcsin(relaxed_scaffold_df["z"]))
relaxed_scaffold_df["lon"] = np.degrees(np.arctan2(relaxed_scaffold_df["y"], relaxed_scaffold_df["x"]))

# Plot the points on a sphere using plotly.
# Display the order names next to the points.
fig = px.scatter_geo(
    relaxed_scaffold_df,
    lat="lat",
    lon="lon",
    hover_name=relaxed_scaffold_df.index,
    text=relaxed_scaffold_df.index,
    title="RELAXED Scaffold Sphere Order Points",
)
fig.update_geos(
    showcountries=False,
    showcoastlines=False,
    showland=False,
    showocean=False,
    showlakes=False,
    showrivers=False,
    projection_type="orthographic",
)
fig.update_layout(margin={"r":0,"t":30,"l":0,"b":0})
fig.update_layout(width=600, height=600)
fig.show()

## Graft order points from 2D projection (tSNE) onto the RELAXED scaffold

Let's see if this works for a single order, Siluriformes


In [ ]:
import pathlib

current_order = 'Siluriformes'  # Change this to visualize a different order

tsne_by_order_output_dir = pathlib.Path('output/tsne_by_order')
    
df_tsne = pd.read_csv(tsne_by_order_output_dir / f"{current_order}_2D_tSNE_sklearn.csv", index_col=0)

# Normalize these points to be between -1 and 1 in both dimensions.
df_tsne['x'] = (df_tsne['x'] - df_tsne['x'].min()) / (df_tsne['x'].max() - df_tsne['x'].min()) * 2 - 1
df_tsne['y'] = (df_tsne['y'] - df_tsne['y'].min()) / (df_tsne['y'].max() - df_tsne['y'].min()) * 2 - 1
# Scale them down quite a bit to fit better on the sphere.
df_tsne['x'] *= 0.3
df_tsne['y'] *= 0.3

# Scale the X up a bit more to account for the sphere's aspect ratio.
df_tsne['x'] *= 1.5

# Find the lat/lon of the center point for this order from the scaffold points.
center_point = relaxed_scaffold_df.loc[current_order]
center_lat = center_point["lat"]
center_lon = center_point["lon"]

# Convert the normalized tSNE points to lat/lon around the center point.
def tsne_to_lat_lon(x, y, center_lat, center_lon):
    # Convert center lat/lon to radians
    center_lat_rad = np.radians(center_lat)
    center_lon_rad = np.radians(center_lon)

    # Calculate the new latitude
    new_lat_rad = np.arcsin(np.sin(center_lat_rad) * np.cos(np.sqrt(x**2 + y**2)) +
                            np.cos(center_lat_rad) * np.sin(np.sqrt(x**2 + y**2)) * (y / np.sqrt(x**2 + y**2)))

    # Calculate the new longitude
    new_lon_rad = center_lon_rad + np.arctan2(x * np.sin(np.sqrt(x**2 + y**2)),
                                              np.cos(np.sqrt(x**2 + y**2)) - np.sin(center_lat_rad) * np.sin(new_lat_rad))

    # Convert back to degrees
    new_lat = np.degrees(new_lat_rad)
    new_lon = np.degrees(new_lon_rad)

    return new_lat, new_lon
df_tsne[['lat', 'lon']] = df_tsne.apply(lambda row: tsne_to_lat_lon(row['x'], row['y'], center_lat, center_lon), axis=1, result_type='expand')

# Plot the tSNE points and the scaffold points on the same sphere.
fig = px.scatter_geo(relaxed_scaffold_df,lat="lat",lon="lon",hover_name=relaxed_scaffold_df.index,title=f"tSNE Points for {current_order} on RELAXED Scaffold Sphere",)
fig.add_scattergeo( lat=df_tsne['lat'],lon=df_tsne['lon'],hovertext=df_tsne.index,mode='markers',marker=dict(size=5, color='red'),name='tSNE Points')
fig.update_geos(showcountries=False,showcoastlines=False,showland=False,showocean=False,showlakes=False,showrivers=False,projection_type="orthographic")
fig.update_layout(margin={"r":0,"t":30,"l":0,"b":0})
fig.update_layout(width=600, height=600)
fig.show()

In [ ]:
# Now convert the lat/lon back to x,y,z coordinates on the unit sphere.
def lat_lon_to_xyz(lat, lon):
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    x = np.cos(lat_rad) * np.cos(lon_rad)
    y = np.cos(lat_rad) * np.sin(lon_rad)
    z = np.sin(lat_rad)
    return x, y, z
df_tsne[['x', 'y', 'z']] = df_tsne.apply(lambda row: lat_lon_to_xyz(row['lat'], row['lon']), axis=1, result_type='expand')

df_tsne.to_csv(tsne_by_order_output_dir / f"{current_order}_3D_tSNE_on_sphere.csv")

In [ ]:
# Load these xyz points back in and plot them in 3D to verify.
df_3d = pd.read_csv(tsne_by_order_output_dir / f"{current_order}_3D_tSNE_on_sphere.csv", index_col=0)
fig = px.scatter_3d(df_3d, x='x', y='y', z='z', color='order', title=f"3D tSNE of {current_order} on Sphere")
fig.update_layout(width=600, height=600)
fig.show()

## Grafting of all orders

Now we do what we did above, but for every order.

There are some exceptions - if the number of points is very small, move them all to be the same as the order-level initial condition point. They will be moved away from each other during the spherical relaxation step.

In [ ]:
# Get all the orders from the scaffold points.
orders = relaxed_scaffold_df.index.tolist()

# We might need mMDS points for some orders that don't have t-SNE results.
mds_by_order_output_dir = pathlib.Path('output/mds_by_order')

for current_order in orders:
    # Load in the t-SNE results for the current order. If the file doesn't exist, we need to handle it
    # differently.
    df_tsne = None
    print(f"Processing order: {current_order}...", end='', flush=True)
    infile = tsne_by_order_output_dir / f"{current_order}_2D_tSNE_sklearn.csv"
    # If the t-SNE file doesn't exist, try the mMDS file.
    if not pd.io.common.file_exists(infile):
        print(f"t-SNE file not found for {current_order}, trying mMDS file...", end='', flush=True)
        infile = mds_by_order_output_dir / f"{current_order}_2D_mMDS_sklearn.csv"

    df_tsne = pd.read_csv(infile, index_col=0)

    # Print the number of genera in this order.
    print(f" {len(df_tsne)} genera...", end='', flush=True)

    # Standard scaling factor for most orders. If the number of genera is small, we use a smaller
    # scaling factor to keep them from being too spread out on the sphere.
    scaling_factor = 0.3
    if len(df_tsne) <= 500 and len(df_tsne) > 10:
        scaling_factor = 0.15
    elif len(df_tsne) <= 10:
        scaling_factor = 0.01

    print(f" scaling factor {scaling_factor}...", end='', flush=True)

    # Normalize these points to be between -1 and 1 on both axes.
    df_tsne['x'] = (df_tsne['x'] - df_tsne['x'].min()) / (df_tsne['x'].max() - df_tsne['x'].min()) * 2 - 1
    df_tsne['y'] = (df_tsne['y'] - df_tsne['y'].min()) / (df_tsne['y'].max() - df_tsne['y'].min()) * 2 - 1

    # Scale them down quite a bit to fit better on the sphere.
    df_tsne['x'] *= scaling_factor
    df_tsne['y'] *= scaling_factor

    # Scale the X up a bit more to account for the sphere's aspect ratio.
    df_tsne['x'] *= 1.5

    # Find the lat/lon of the center point for this order from the scaffold points.
    center_point = relaxed_scaffold_df.loc[current_order]
    center_lat = center_point["lat"]
    center_lon = center_point["lon"]

    # Convert the normalized t-SNE points to lat/lon around the center point.
    # Note that this relies on tsne_to_lat_lon being defined above.
    df_tsne[['lat', 'lon']] = df_tsne.apply(lambda row: tsne_to_lat_lon(row['x'], row['y'], center_lat, center_lon), axis=1, result_type='expand')

    # Convert the lat/lon to x,y,z coordinates on the unit sphere. Note that
    # this relies on lat_lon_to_xyz being defined above.
    df_tsne[['x', 'y', 'z']] = df_tsne.apply(lambda row: lat_lon_to_xyz(row['lat'], row['lon']), axis=1, result_type='expand')

    df_tsne.to_csv(tsne_by_order_output_dir / f"{current_order}_3D_on_sphere.csv")
    print("done.")

In [ ]:
# Merge all the orders into one big file.
orders_to_merge = orders
merged_df = pd.DataFrame()

merged_filename_prefix = f"{tsne_by_order_output_dir}/Actinopterygii_relax100_scaffold_tSNE_3D_merged"

merged_filename = f"{merged_filename_prefix}.csv"

for current_order in orders_to_merge:
    df_3d = pd.read_csv(f"{tsne_by_order_output_dir}/{current_order}_3D_on_sphere.csv", index_col=0)

    # If the order has only one point, use the point from the scaffold as the
    # point for this genus. The relaxed_scaffold_df also has lat and lon, but we won't
    # use these since we only really need the xyz coordinates for the tree
    # integration step, and the lat/lon are just intermediate values for
    # calculating the xyz coordinates.
    if len(df_3d) == 1:
        print(f"Order {current_order} has only one point. Using scaffold point for this order.")
        scaffold_point = relaxed_scaffold_df.loc[current_order]
        df_3d.at[df_3d.index[0], 'x'] = scaffold_point['x']
        df_3d.at[df_3d.index[0], 'y'] = scaffold_point['y']
        df_3d.at[df_3d.index[0], 'z'] = scaffold_point['z']

    merged_df = pd.concat([merged_df, df_3d], axis=0)

merged_df.to_csv(merged_filename)

print(f"\n\nAll orders merged into {merged_filename}")

## Viz and relaxation of all orders

Now we run relaxation to better spread the points out on the sphere. This uses Wandrille's `spherical-scatterplot-relaxation` script.

In [ ]:
fig = px.scatter_3d(merged_df, x='x', y='y', z='z', color='order', 
                    title="All Actinopterygii tSNE grafted onto relaxed (100)order scaffold Sphere")
fig.update_layout(width=800, height=800)
fig.show()


In [ ]:
# Get some paths ready for running the relaxation script on the 3D tSNE points on
# the sphere. This will help spread them out more evenly on the sphere and make
# them look nicer.
#
# These paths are also used below.

import os

relaxation_output_dir = f"{tsne_by_order_output_dir}/Actinopterygii_relax100_scaffold_tSNE_relax"

os.makedirs(relaxation_output_dir, exist_ok=True)

relaxtion_cmd = "./spherical-scatterplot-relaxation/spherical_scatterplot_relaxation.py "
relaxtion_cmd += f"-i {merged_filename} "

relaxation_output_dir_and_prefix = f"{relaxation_output_dir}/Actinopterygii_relax100_scaffold_tSNE_relax"
relaxtion_cmd += f"-o {relaxation_output_dir_and_prefix} "
relaxtion_cmd += "-n 500 -n 100 -w 5 -l 0.05"


In [ ]:
# This takes about 10 minutes on Alienware Area-51m.
%run $relaxtion_cmd

In [ ]:
# The '_round' inserted into the filename makes things pretty verbose later on. Let's remove that from all the filenames in the relaxation output directory.
for filename in os.listdir(relaxation_output_dir):
    if filename.endswith(".csv") and "_round" in filename:
        new_filename = filename.replace("_round", "")
        os.rename(os.path.join(relaxation_output_dir, filename), os.path.join(relaxation_output_dir, new_filename))
    

In [ ]:
# Use a slider to visualize the relaxation process.
fig = go.Figure()
rounds_to_plot = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]
# Add traces, one for each slider step
for step in range(len(rounds_to_plot)):
    filename = f"{relaxation_output_dir_and_prefix}{rounds_to_plot[step]}.csv"

    # reading data for that round
    tmp = pd.read_csv( filename , index_col= 0 )

    fig.add_trace(
        go.Scatter3d(
            x=tmp['x'],
            y=tmp['y'],
            z=tmp['z'],
            mode='markers',
            marker=dict(size=3, color=tmp['order'].astype('category').cat.codes, colorscale='Viridis', opacity=0.8),
            hovertext=tmp.order,
            name=f'Round {rounds_to_plot[step]}',
            visible=(step == 0)  # Only the first trace is visible initially
        )
    )
# Create slider steps
steps = []
for i in range(len(rounds_to_plot)):
    step = dict(
        method="update",
        args=[{"visible": [False] * len(rounds_to_plot)},
              {"title": f"3D tSNE of All Actinopterygii on relaxed (100) Sphere - Relaxation Round {rounds_to_plot[i]}"}],  # layout attribute
    )
    step["args"][0]["visible"][i] = True  # Toggle i'th trace to "visible"
    steps.append(step)
sliders = [dict(
    active=0,
    currentvalue={"prefix": "Round: "},
    pad={"t": 50},
    steps=steps
)]
fig.update_layout(
    sliders=sliders,
    width=800,
    height=800,
    # We start on round 0. The title gets updated by the slider steps, but we
    # need to set an initial title here too.
    title="3D tSNE of All Actinopterygii on relaxed (100) Sphere - Relaxation Round 0"
)
fig.show()

Round 100 looks pretty good... now to import into OpenSpace...

In [ ]:
round_number_to_keep = 100

relaxation_keep_file = f"{relaxation_output_dir_and_prefix}{round_number_to_keep}.csv"

# Quick check. How many unique families total?
df_final = pd.read_csv(relaxation_keep_file, index_col=0)
print(f"Total genera: {len(df_final)}")
# Count number of unique families
unique_families = df_final['family'].nunique()
print(f"Total families: {unique_families}")

In [ ]:
# Find points with NaN x, y, or z values.
nan_points = df_final[df_final[['x', 'y', 'z']].isnull().any(axis=1)]
if len(nan_points) > 0:
    print("Warning: Found points with NaN x, y, or z values in the dataset.")
    print(nan_points)

How many duplicate points? This should match what OpenSpace reports.

In [ ]:
duplicate_xyz = df_final[df_final.duplicated(subset=["x", "y", "z"], keep=False)]
if len(duplicate_xyz) > 0:
    print("Warning: Found duplicate x, y, z values in the dataset.")
    print(duplicate_xyz)

Looks ok, let's drop lat and lon columns and save it to base output dir. 

In [ ]:
# Drop lat and lon columns since we don't need them anymore. Check first to see if they are there.
if 'lat' in df_final.columns and 'lon' in df_final.columns:
    df_final = df_final.drop(columns=['lat', 'lon'])

# Get just the base filename without the directory
final_relaxed_filename = f"output/{os.path.basename(relaxation_keep_file)}"

df_final.to_csv(final_relaxed_filename)

print(f"Final relaxed tSNE points on relaxed scaffold saved to {final_relaxed_filename}")

## Re-making scaffold points

The original scaffold points were used to provide initial positions for the order-level points. We've moved these around, so now we need to re-compute the shifted positions. To do this, on an order-by-order basis, we'll compute the centroid and use that as the location of the order-level point. We'll then export the new scaffold to run through Wandrille's `integrate_tree_to_xyz` script.


In [ ]:
# Columns for the new df are order, x, y, z.
new_scaffold_df = pd.DataFrame(columns=['order', 'x', 'y', 'z'])

# Recall that 'orders' is loaded at the top from the scaffold points, and should
# represent every order.
for order in orders:
    if order in df_final['order'].values:
        subset = df_final[df_final['order'] == order]
        centroid = subset[['x', 'y', 'z']].mean()
        # What's the radius of this centroid?
        radius = np.sqrt(centroid['x']**2 + centroid['y']**2 + centroid['z']**2)
        # Normalize the centroid to be on the unit sphere.
        centroid['x'] /= radius
        centroid['y'] /= radius
        centroid['z'] /= radius
        # print this out.
        new_scaffold_df.loc[len(new_scaffold_df)] = {'order': order, 'x': centroid['x'], 'y': centroid['y'], 'z': centroid['z']}
    else:
        print(f"Order {order} not found in points, skipping.")

# The prefix is useful below for writing the branch filenames below. Take the
# final relaxed filename and remove the .csv extension to get the prefix. Then
# append "_scaffold" to the prefix for the final scaffold filename.
tsne_final_scaffold_output_pathname_prefix = final_relaxed_filename.replace(".csv", "") + "_scaffold"
tsne_final_scaffold_output_filename = f"{tsne_final_scaffold_output_pathname_prefix}.csv"

new_scaffold_df.to_csv(tsne_final_scaffold_output_filename, index=False)

print(f"Final scaffold points saved to {tsne_final_scaffold_output_filename}")

## The Tree

Finally, Wandrille's tree script to make branches.

In [ ]:
tree_filename = "output\\cleaned_trees\\Actinopterygii_orders.nwk"

integrate_tree_command = f"./integrate_tree_to_XYZ/integrate_tree_to_XYZ.py "
integrate_tree_command += f"-i {tsne_final_scaffold_output_filename} "
integrate_tree_command += f"-t {tree_filename} "
integrate_tree_command += f"-o {tsne_final_scaffold_output_pathname_prefix} "
integrate_tree_command += "--ignore-missing --use-z-from-file"

%run $integrate_tree_command